# Workshop 3W — Upsell Like an Engineer
    Unit 3 — Programming for Intelligent Products

    Today we’ll practise TDD (Red → Green → Refactor), experiment logging, and debugging discipline.

In [2]:

import sys, platform
print("Python:", sys.version)
print("Platform:", platform.platform())
import mlflow, pandas as pd
print("MLflow:", mlflow.__version__)
print("Environment ready ✅")

Python: 3.14.2 (main, Sep 10 2026, 12:33:04) [GCC 13.3.0]
Platform: Linux-6.8.0-1064-azure-x86_64-with-glibc2.39
MLflow: 3.16.1
Environment ready ✅


In [20]:
def should_upsell(order: dict) -> bool:
    """Predict if a customer will buy a drink."""
    if order.get("order_size", 0) >= 4:
        return True
    
    if order.get("loyalty_member") == "yes":
        if order.get("time_of_day") == "lunch":
            return True
        elif order.get("time_of_day") == "evening":
            return False
    else:
        if order.get("time_of_day") == "lunch":
            return False
        elif order.get("time_of_day") == "evening" and order.get("temperature", 0) >= 32:
            return True    
    
    return None  # start with a failing stub (RED)


# Positive test – loyalty at lunch → True
order_1a = {"time_of_day": "lunch", "loyalty_member": "yes"}
assert should_upsell(order_1a) is True, "FAIL: Loyalty at lunch should be upsold."
print("PASS: Test 1a")

# Negative test – non-loyalty at lunch → False (prevents `return True`)
order_1b = {"time_of_day": "lunch", "loyalty_member": "no"}
assert should_upsell(order_1b) is False, "FAIL: Non-loyalty at lunch should NOT be upsold."
print("PASS: Test 1b")

PASS: Test 1a
PASS: Test 1b


In [21]:
# Evening loyalty → should NOT be upsold
order_2 = {"time_of_day": "evening", "loyalty_member": "yes"}
assert should_upsell(order_2) is False, "FAIL: Evening loyalty should NOT be upsold."
print("PASS: Test 2")

PASS: Test 2


In [15]:
try:
    import mlflow
    mlflow.set_tracking_uri("sqlite:///mlflow.db")
    mlflow.set_experiment("hotdog-upsell")
    with mlflow.start_run(run_name="Initial Loyalty Rule"):
        order = {"time_of_day": "lunch", "loyalty_member": "yes"}
        prediction = should_upsell(order)
        mlflow.log_params(order)
        mlflow.log_metric("prediction", int(bool(prediction)))
        mlflow.set_tag("rule_version", "v1")
    print("Experiment logged successfully ✅")
except Exception:
    print("[INFO] MLflow not installed — skipping logging safely.")

Experiment logged successfully ✅


In [22]:
# Heatwave (temperature 30 or greater) → True
order_3 = {"time_of_day": "evening", "loyalty_member": "no", "temperature": 32}
assert should_upsell(order_3) is True, "FAIL: Hot weather should trigger upsell."
print("PASS: Test 3")

# Large order (order size 4 or greater) → True
order_4 = {"order_size": 4}
assert should_upsell(order_4) is True, "FAIL: Large orders should trigger upsell."
print("PASS: Test 4")

PASS: Test 3
PASS: Test 4


In [23]:
# Busy lunch heatwave (non-loyalty) → should NOT upsell
order_5 = {"time_of_day": "lunch", "loyalty_member": "no", "temperature": 32}
assert should_upsell(order_5) is False, "FAIL: Busy lunch heatwave (non-loyalty) should NOT upsell."
print("PASS: Test 5")

PASS: Test 5


In [24]:
print("### My Commit Log")
commits = [
    "feat: Add initial failing test for lunch loyalty",
    "fix: Implement lunch loyalty rule",
    "feat: Add evening loyalty test",
    "fix: Handle time-of-day logic",
    "refactor: Simplify function and add docstring",
    "fix: Add busy heatwave override",
]
for c in commits:
    print("-", c)

### My Commit Log
- feat: Add initial failing test for lunch loyalty
- fix: Implement lunch loyalty rule
- feat: Add evening loyalty test
- fix: Handle time-of-day logic
- refactor: Simplify function and add docstring
- fix: Add busy heatwave override
